In [1]:
SYMBOL = "BTCUSDT"
TARGET_HORIZON = 5
INTERVAL = 1
MODEL_TYPE = "rf"

In [2]:
# Parameters
SYMBOL = "BNBUSDT"
INTERVAL = "5m"
TARGET_HORIZON = 6
MODEL_TYPE = "rf"


In [3]:
import os
import time
import json
import joblib
import pandas as pd
import numpy as np
import optuna
from optuna.pruners import MedianPruner
from functools import partial
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss,
    brier_score_loss,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)
from features import add_features
from constants import DATA_DIR, MODEL_DIR
from utils import time_split, information_coefficient, rank_information_coefficient
from models import OBJECTIVES, MODEL_REGISTRY

/home/rachmiel/quant/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
MODEL_DIR = os.path.join(MODEL_DIR, MODEL_TYPE)
PARQUET_PATH = f"{DATA_DIR}/{SYMBOL}_{INTERVAL}.parquet"

os.makedirs(MODEL_DIR, exist_ok=True)

In [5]:
model_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_model.joblib")
features_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_feature_cols.json")
meta_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_meta.json")
fi_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_feature_importance.csv")
pred_path = os.path.join(MODEL_DIR, f"{SYMBOL}__{TARGET_HORIZON}_predictions.csv")

In [6]:
df = pd.read_parquet(PARQUET_PATH)
print(f"[info] raw rows: {len(df):,}")

# add features + target
df, feature_cols = add_features(df, TARGET_HORIZON)

[info] raw rows: 83,520


In [7]:
df.head()

,open_time,open,high,low,close,volume,close_time,quote_asset_volume,num_trades,taker_buy_base_asset_volume,...,dow_cos,dom_sin,dom_cos,month_sin,month_cos,macd,macd_signal,macd_hist,atr_14,atr_norm
0,2025-06-01 00:00:00+00:00,658.01,658.08,657.61,657.61,403.616,2025-06-01 00:04:59.999999+00:00,265524.57169,2043,174.587,...,0.62349,0.201299,0.97953,1.224647e-16,-1.0,0.000000,0.000000,0.000000,NaN,NaN
1,2025-06-01 00:05:00+00:00,657.61,657.91,657.48,657.90,235.687,2025-06-01 00:09:59.999999+00:00,155007.00488,1438,123.239,...,0.62349,0.201299,0.97953,1.224647e-16,-1.0,0.006506,0.003615,0.002892,NaN,NaN
2,2025-06-01 00:10:00+00:00,657.90,658.08,657.12,657.28,517.657,2025-06-01 00:14:59.999999+00:00,340365.13877,1673,336.061,...,0.62349,0.201299,0.97953,1.224647e-16,-1.0,-0.010936,-0.002349,-0.008587,NaN,NaN
3,2025-06-01 00:15:00+00:00,657.28,657.40,656.80,656.90,335.908,2025-06-01 00:19:59.999999+00:00,220733.77620,1928,131.947,...,0.62349,0.201299,0.97953,1.224647e-16,-1.0,-0.032320,-0.012502,-0.019819,NaN,NaN
4,2025-06-01 00:20:00+00:00,656.89,657.43,656.10,656.71,1482.819,2025-06-01 00:24:59.999999+00:00,973507.37841,3894,291.438,...,0.62349,0.201299,0.97953,1.224647e-16,-1.0,-0.050821,-0.023901,-0.026920,NaN,NaN


In [8]:
target_col = f"target_{TARGET_HORIZON}"
ret_col = f"target_ret_fwd_{TARGET_HORIZON}"

model_df = df[["open_time"] + feature_cols + [target_col, ret_col]].copy()

# Remove:
# early rows where rolling features don’t exist yet
# rows where z-scores / ratios blew up
# rows where target is NaN (due to future shift)
model_df = model_df.replace([np.inf, -np.inf], np.nan)
model_df = model_df.dropna(subset=feature_cols + [target_col, ret_col])

print(f"[info] usable rows after features: {len(model_df):,}")

train_df, test_df = time_split(model_df, train_frac=0.8)

# Further split the training set into train/valid for Optuna
optuna_train_df, valid_df = time_split(train_df, train_frac=0.8)

X_train = optuna_train_df[feature_cols]
y_train = optuna_train_df[target_col]

X_valid = valid_df[feature_cols]
y_valid = valid_df[target_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]
fwd_ret = test_df[ret_col]

train_start_time = pd.to_datetime(train_df["open_time"].iloc[0], utc=True)
train_end_time = pd.to_datetime(train_df["open_time"].iloc[-1], utc=True)

val_start_time = pd.to_datetime(valid_df["open_time"].iloc[0], utc=True)
val_end_time = pd.to_datetime(valid_df["open_time"].iloc[-1], utc=True)

test_start_time = pd.to_datetime(test_df["open_time"].iloc[0], utc=True)
test_end_time = pd.to_datetime(test_df["open_time"].iloc[-1], utc=True)

print(f"[info] optuna train rows: {len(optuna_train_df):,}")
print(f"[info] valid rows:        {len(valid_df):,}")
print(f"[info] test rows:         {len(test_df):,}")

[info] usable rows after features: 83,441
[info] optuna train rows: 53,401
[info] valid rows:        13,351
[info] test rows:         16,689


In [9]:
pruner = MedianPruner(n_warmup_steps=5, n_min_trials=10)
study = optuna.create_study(direction="maximize", pruner=pruner)

class EarlyStoppingCallback:
    def __init__(self, patience: int):
        self.patience = patience
        self.best_value = -float('inf')
        self.no_improvement_count = 0

    def __call__(self, study, trial):
        if study.best_value > self.best_value:
            self.best_value = study.best_value
            self.no_improvement_count = 0
        else:
            self.no_improvement_count += 1

        if self.no_improvement_count >= self.patience:
            study.stop()

early_stopping = EarlyStoppingCallback(patience=10)

objective_fn = partial(
    OBJECTIVES[MODEL_TYPE],
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
)

study.optimize(objective_fn, n_trials=50, callbacks=[early_stopping], show_progress_bar=True)

print("\n[optuna] best trial")
print(f"value: {study.best_value:.6f}")
print("params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

[I 2026-03-20 15:39:43,427] A new study created in memory with name: no-name-bd223ae7-dab6-4283-9eae-d0830458ac00


  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:11<?, ?it/s]

Best trial: 0. Best value: 0.523031:   0%|          | 0/50 [00:11<?, ?it/s]

Best trial: 0. Best value: 0.523031:   2%|▏         | 1/50 [00:11<09:20, 11.44s/it]

...

Best trial: 12. Best value: 0.537537:  44%|████▍     | 22/50 [02:57<03:42,  7.93s/it]

Best trial: 12. Best value: 0.537537:  44%|████▍     | 22/50 [02:57<03:42,  7.93s/it]

Best trial: 12. Best value: 0.537537:  46%|████▌     | 23/50 [02:57<02:55,  6.48s/it]

Best trial: 12. Best value: 0.537537:  46%|████▌     | 23/50 [02:57<03:28,  7.73s/it]

[I 2026-03-20 15:42:41,138] Trial 22 finished with value: 0.5355144100172324 and parameters: {'n_estimators': 700, 'max_depth': 10, 'min_samples_split': 2, 'min_samples_leaf': 13, 'max_features': 'log2', 'bootstrap': False, 'class_weight': 'balanced'}. Best is trial 12 with value: 0.5375372951983405.

[optuna] best trial
value: 0.537537
params:
  n_estimators: 600
  max_depth: 10
  min_samples_split: 2
  min_samples_leaf: 14
  max_features: 0.3
  bootstrap: False
  class_weight: balanced


In [10]:
best_params = study.best_params.copy()
best_params["random_state"] = 42
best_params["n_jobs"] = -1

X_train_full = train_df[feature_cols]
y_train_full = train_df[target_col]

final_model = MODEL_REGISTRY[MODEL_TYPE](**best_params)

start = time.time()
print(f"[training] fitting final {MODEL_TYPE}...")
final_model.fit(X_train_full, y_train_full)
print(f"[training] done in {time.time() - start:.2f}s")

[training] fitting final rf...


[training] done in 4.75s


In [11]:
train_pred = final_model.predict_proba(X_train_full)[:, 1]
test_pred = final_model.predict_proba(X_test)[:, 1]

In [12]:
train_pred_label = (train_pred >= 0.5).astype(int)
test_pred_label = (test_pred >= 0.5).astype(int)

print("[eval] computing metrics...")

train_auc = roc_auc_score(y_train_full, train_pred)
test_auc = roc_auc_score(y_test, test_pred)

train_pr_auc = average_precision_score(y_train_full, train_pred)
test_pr_auc = average_precision_score(y_test, test_pred)

train_logloss = log_loss(y_train_full, np.clip(train_pred, 1e-8, 1 - 1e-8))
test_logloss = log_loss(y_test, np.clip(test_pred, 1e-8, 1 - 1e-8))

train_brier = brier_score_loss(y_train_full, train_pred)
test_brier = brier_score_loss(y_test, test_pred)

train_acc = accuracy_score(y_train_full, train_pred_label)
test_acc = accuracy_score(y_test, test_pred_label)

train_precision = precision_score(y_train_full, train_pred_label, zero_division=0)
test_precision = precision_score(y_test, test_pred_label, zero_division=0)

train_recall = recall_score(y_train_full, train_pred_label, zero_division=0)
test_recall = recall_score(y_test, test_pred_label, zero_division=0)

train_f1 = f1_score(y_train_full, train_pred_label, zero_division=0)
test_f1 = f1_score(y_test, test_pred_label, zero_division=0)

print("\n===== RESULTS =====")
print(f"Train ROC AUC:   {train_auc:.6f}")
print(f"Test ROC AUC:    {test_auc:.6f}")
print(f"Train PR AUC:    {train_pr_auc:.6f}")
print(f"Test PR AUC:     {test_pr_auc:.6f}")
print(f"Train Log Loss:  {train_logloss:.6f}")
print(f"Test Log Loss:   {test_logloss:.6f}")
print(f"Train Brier:     {train_brier:.6f}")
print(f"Test Brier:      {test_brier:.6f}")
print(f"Train Accuracy:  {train_acc:.6f}")
print(f"Test Accuracy:   {test_acc:.6f}")
print(f"Train Precision: {train_precision:.6f}")
print(f"Test Precision:  {test_precision:.6f}")
print(f"Train Recall:    {train_recall:.6f}")
print(f"Test Recall:     {test_recall:.6f}")
print(f"Train F1:        {train_f1:.6f}")
print(f"Test F1:         {test_f1:.6f}")

[eval] computing metrics...

===== RESULTS =====
Train ROC AUC:   0.867555
Test ROC AUC:    0.529629
Train PR AUC:    0.879741
Test PR AUC:     0.518604
Train Log Loss:  0.632591
Test Log Loss:   0.693167
Train Brier:     0.220100
Test Brier:      0.249980
Train Accuracy:  0.773670
Test Accuracy:   0.516208
Train Precision: 0.761603
Test Precision:  0.508497
Train Recall:    0.814820
Test Recall:     0.599223
Train F1:        0.787313
Test F1:         0.550145


In [13]:
eval_df = pd.DataFrame({
    "pred": test_pred,
    "y_cls": y_test.values,
    "fwd_ret": fwd_ret.values,   # continuous realised return
})

eval_df["pred_bin"] = pd.qcut(eval_df["pred"], 10, duplicates="drop")
bucket_stats = eval_df.groupby("pred_bin")["fwd_ret"].agg(["mean", "count", "std"])
print(bucket_stats)

                    mean  count       std
pred_bin                                 
(0.248, 0.449] -0.000211   1669  0.004032
(0.449, 0.47]  -0.000004   1669  0.003965
(0.47, 0.485]  -0.000310   1669  0.004335
(0.485, 0.498] -0.000001   1669  0.003924
(0.498, 0.509] -0.000393   1669  0.004477
(0.509, 0.519] -0.000146   1668  0.004455
(0.519, 0.528] -0.000130   1669  0.004214
(0.528, 0.538] -0.000092   1669  0.004158
(0.538, 0.553]  0.000095   1669  0.004407
(0.553, 0.771]  0.000150   1669  0.006637


/tmp/ipykernel_296694/1883822384.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  bucket_stats = eval_df.groupby("pred_bin")["fwd_ret"].agg(["mean", "count", "std"])


In [14]:
top_decile_threshold = float(np.quantile(test_pred, 0.9))
bottom_decile_threshold = float(np.quantile(test_pred, 0.1))

top_decile_mean_ret = float(eval_df.loc[eval_df["pred"] >= top_decile_threshold, "fwd_ret"].mean())
bottom_decile_mean_ret = float(eval_df.loc[eval_df["pred"] <= bottom_decile_threshold, "fwd_ret"].mean())
overall_mean_ret = float(eval_df["fwd_ret"].mean())

signal_threshold = 0.6
signal_rate = float((eval_df["pred"] >= signal_threshold).mean())
signal_mean_ret = float(eval_df.loc[eval_df["pred"] >= signal_threshold, "fwd_ret"].mean())

In [15]:
# feature importance
importances = pd.Series(
    final_model.feature_importances_,
    index=feature_cols
).sort_values(ascending=False)

print("\n===== FEATURE IMPORTANCE =====")
print(importances)


===== FEATURE IMPORTANCE =====
vol_30              0.055973
mom_60              0.045891
mom_30              0.045398
vol_regime_ratio    0.044092
imbalance_15        0.041340
hour_cos            0.041143
dom_sin             0.038641
vol_15              0.038260
range_15            0.036700
trend_strength      0.036693
atr_norm            0.033677
hour_sin            0.033167
dist_ma_30          0.030866
dom_cos             0.030317
macd_hist           0.030077
month_cos           0.026322
range_5             0.025560
mom_15              0.023337
range_ratio         0.023042
dow_sin             0.021330
vol_5               0.021281
vol_ratio_5_30      0.021042
imbalance_5         0.019787
dist_ma_15          0.019571
trend_x_imb         0.019487
dist_ma_15_z        0.019311
mr_x_vol            0.019100
mom_10              0.018627
mom_5               0.016087
mom_x_imb           0.015007
dow_cos             0.013470
dist_ma_5           0.012329
month_sin           0.012073
trades_z   

In [16]:
# save predictions
out = test_df[["open_time", target_col]].copy()
out["prediction"] = test_pred
out.to_csv(pred_path, index=False)
print(f"\n[saved] predictions -> {pred_path}")


[saved] predictions -> models/rf/BNBUSDT__6_predictions.csv


In [17]:
# save model
joblib.dump(final_model, model_path)

# save feature columns
with open(features_path, "w") as f:
    json.dump(feature_cols, f, indent=2)

# save feature importance
importances.to_csv(fi_path, header=["importance"])

# save metadata
meta = {
    "symbol": SYMBOL,
    "target_horizon": int(TARGET_HORIZON),
    "target_col": target_col,
    "model_type": MODEL_TYPE,
    "study_best_value": float(study.best_value),
    "model_params": best_params,
    "n_features": int(len(feature_cols)),
    "feature_cols_path": str(features_path),
    "model_path": str(model_path),
    "feature_importance_path": str(fi_path) if fi_path is not None else None,
    "train_auc": float(train_auc),
    "test_auc": float(test_auc),
    "train_pr_auc": float(train_pr_auc),
    "test_pr_auc": float(test_pr_auc),
    "train_logloss": float(train_logloss),
    "test_logloss": float(test_logloss),
    "train_brier": float(train_brier),
    "test_brier": float(test_brier),
    "train_accuracy": float(train_acc),
    "test_accuracy": float(test_acc),
    "train_precision": float(train_precision),
    "test_precision": float(test_precision),
    "train_recall": float(train_recall),
    "test_recall": float(test_recall),
    "train_f1": float(train_f1),
    "test_f1": float(test_f1),
    "test_top_decile_threshold": top_decile_threshold,
    "test_bottom_decile_threshold": bottom_decile_threshold,
    "test_top_decile_mean_fwd_ret": top_decile_mean_ret,
    "test_bottom_decile_mean_fwd_ret": bottom_decile_mean_ret,
    "test_overall_mean_fwd_ret": overall_mean_ret,
    "test_signal_threshold": signal_threshold,
    "test_signal_rate": signal_rate,
    "test_signal_mean_fwd_ret": signal_mean_ret,
    "train_start_time": pd.Timestamp(train_start_time).isoformat(),
    "train_end_time": pd.Timestamp(train_end_time).isoformat(),
    "val_start_time": pd.Timestamp(val_start_time).isoformat(),
    "val_end_time": pd.Timestamp(val_end_time).isoformat(),
    "test_start_time": pd.Timestamp(test_start_time).isoformat(),
    "test_end_time": pd.Timestamp(test_end_time).isoformat(),
    "train_positive_rate": float(y_train_full.mean()),
    "test_positive_rate": float(y_test.mean()),
    "test_pred_mean": float(np.mean(test_pred)),
    "test_pred_std": float(np.std(test_pred)),
    "test_pred_p10": float(np.quantile(test_pred, 0.10)),
    "test_pred_p50": float(np.quantile(test_pred, 0.50)),
    "test_pred_p90": float(np.quantile(test_pred, 0.90)),
}

with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print(f"[saved] model -> {model_path}")
print(f"[saved] features -> {features_path}")
print(f"[saved] feature importance -> {fi_path}")
print(f"[saved] metadata -> {meta_path}")

[saved] model -> models/rf/BNBUSDT__h6_model.joblib
[saved] features -> models/rf/BNBUSDT__h6_feature_cols.json
[saved] feature importance -> models/rf/BNBUSDT__h6_feature_importance.csv
[saved] metadata -> models/rf/BNBUSDT__h6_meta.json
